# 07 · 低换手 RL 季度滚动分析

只读取服务器汇总包；不重训、不加载 checkpoint、不读取数据库。先用 `analyze-bundle` 校验并解包，或直接指向服务器复制来的 `summary/`。所有收益来自历史开发验证，成本为模拟情景。


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

SUMMARY_DIR = Path("填写解包后的汇总目录")
if not (SUMMARY_DIR / "summary.json").is_file():
    raise ValueError("请将 SUMMARY_DIR 指向含 summary.json 的结果目录；只有 screen 阶段的包尚无季度结果。")
status = json.loads((SUMMARY_DIR / "status.json").read_text())
if status["status"] != "success" or status["stage"] == "screen":
    raise ValueError("失败或仅筛选阶段的运行不能作为完整季度实验分析。")
summary = json.loads((SUMMARY_DIR / "summary.json").read_text())
config = json.loads((SUMMARY_DIR / "config.json").read_text())
metrics = pd.DataFrame(summary["metrics"]).T
quarters = pd.DataFrame(summary["quarter_returns"])
display(metrics[["total_return", "max_drawdown", "annualized_volatility", "mean_turnover", "total_fees"]])


## 研究门槛与参数冻结

`research_pass` 只意味着值得前向模拟，不意味着可以上线。注意 alpha_sample_status；2025 已观察过，不能重新包装成盲测。先看三个种子分布，再看平均收益。


In [ ]:
display(pd.DataFrame(summary["seed_dispersion"]).T)
for arm, result in summary["judgments"].items():
    print(arm, "research_pass:", result["research_pass"], "production_ready:", result["production_ready"])
    display(pd.Series(result["criteria"], name="是否满足"))
    print("区块重采样：", result["bootstrap"])
display(pd.DataFrame(json.loads((SUMMARY_DIR / "selected.json").read_text())).T)


## 连续净值：滚动训练与固定模型

以下选择同一成本、同一种子。季度换模型继承实际持仓，图中不将每季度净值重新归一为 1。


In [ ]:
SEED = config["seeds"][0]
COST_BP = config["primary_cost_bp"]
keys = [f"robust/{COST_BP:g}"] + [f"{arm}/{SEED}/{COST_BP:g}/{mode}" for arm in ("original", "penalty", "retention") for mode in ("rolling", "fixed")]
fig, ax = plt.subplots(figsize=(12, 5))
for key in keys:
    df = pd.read_csv(SUMMARY_DIR / (key.replace("/", "_") + ".csv"))
    ax.plot(pd.to_datetime(df.return_end_date.astype(str)), df.nav, label=key)
ax.set(title="Continuous NAV — same seed and cost scenario", xlabel="Settlement date", ylabel="NAV (initial = 1)")
ax.legend(fontsize=8); ax.grid(alpha=.2)
plt.show()


## 季度稳定性与成本压力

季度收益按逐日净收益复利；累计费用为金额/初始净值，不是年费率。额外 bp 是每单位买卖成交额的模拟摩擦，各情景均重新运行执行与推理。


In [ ]:
primary = f"{config['primary_cost_bp']:g}"
relative = pd.DataFrame({arm: quarters[[f"{arm}/{seed}/{primary}/rolling" for seed in config["seeds"]]].median(axis=1) - quarters[f"robust/{primary}"] for arm in ("original", "penalty", "retention")})
(relative * 100).plot.bar(figsize=(12, 4), title="Quarterly median return difference vs robust (pp)")
plt.ylabel("Percentage points"); plt.xlabel("Settlement quarter"); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8, 4))
for arm in ("original", "penalty", "retention"):
    values = [metrics.loc[[f"{arm}/{seed}/{float(cost):g}/rolling" for seed in config["seeds"]], "total_return"].median()*100 for cost in config["costs_bp"]]
    ax.plot(config["costs_bp"], values, marker="o", label=arm)
ax.plot(config["costs_bp"], [metrics.loc[f"robust/{float(cost):g}", "total_return"]*100 for cost in config["costs_bp"]], marker="o", label="robust")
ax.set(xlabel="Additional cost per traded notional (bp)", ylabel="Cumulative return (%)", title="Cost stress — median over seeds")
ax.legend(); ax.grid(alpha=.2); plt.show()


## 环境与设备效率

可选：从服务器复制 `profile.json`。环境耗时使用相同动作和多次重复；网络训练计时与环境微基准不同，不能混为同一个加速比。


In [ ]:
PROFILE_PATH = None  # 例如 Path("/path/to/profile.json")
if PROFILE_PATH is not None:
    profile = json.loads(Path(PROFILE_PATH).read_text())
    display(profile["environment"])
    display(profile["training"])
    print("预算估计应使用同一服务器测量；不能把本机 CPU 提速直接套用到 4090。")
